# 01_Preprocess_And_Upload_Gemma_Multimodal_Dataset

이 노트북은 다음 작업을 수행합니다.
1. 필라테스 이미지 폴더를 읽어서 이미지-텍스트 샘플 생성
2. Alpaca 형식 텍스트 JSON(`instruction`, `input`, `output`) 로드
3. 두 데이터를 하나의 Hugging Face Dataset으로 통합
4. Hub에 업로드

**권장 폴더 구조 예시**
image_samples 내 이름은 이미지의 라벨링 이름으로 설정가능
```
dataset/04. 멀티모달 샘플/image_samples/
  hundred/*.png
  teaser/*.png
  swan/*.png
  bridge/*.png
  plank/*.png

/dataset/text_samples/fintech_dataset.json
```

### 라이브러리 선언

In [8]:
# ============================================================
# 라이브러리 임포트
# ============================================================

# === 표준 라이브러리 ===
import os
import random

# === 데이터 처리 ===
import pandas as pd
from PIL import Image

# === HuggingFace ===
from datasets import Dataset, Features, Value, concatenate_datasets
from huggingface_hub import login

# === 환경변수 ===
from dotenv import load_dotenv

## colab 환경여부 확인

In [9]:
# Colab 환경이면 Drive 마운트
try:
    from google.colab import drive, userdata
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/gdrive')
print('IN_COLAB =', IN_COLAB)

IN_COLAB = False


### 기본 설정 허깅페이스 및 이미지 확장자 등

In [10]:
# .env 파일 로드 (override=True: .env 수정 후 재실행 시 항상 반영)
load_dotenv(override=True)

# ===== HuggingFace 설정 =====
hf_token        = os.getenv("HF_TOKEN", "")
HF_DATASET_REPO = os.getenv("HF_DATASET_REPO", "hyokwan/multi_modal_sample")

# ===== 데이터 경로 (.env 변경으로 대상 경로 교체 가능) =====
IMAGE_ROOT      = os.getenv("IMAGE_ROOT",      "./dataset/04. 멀티모달 샘플/image_samples")
IMAGE_META_ROOT = os.getenv("IMAGE_META_ROOT", "./dataset/04. 멀티모달 샘플/image_samples/labels.csv")
TEXT_JSON_PATH  = os.getenv("TEXT_JSON_PATH",  "./dataset/04. 멀티모달 샘플/text_samples/sample_data.json")

# ===== 이미지 수집 설정 =====
IMAGE_EXTS  = {'.jpg', '.jpeg', '.png', '.webp'}
_maxSamples = os.getenv("MAX_IMAGE_SAMPLES_PER_CLASS", "")
MAX_IMAGE_SAMPLES_PER_CLASS = int(_maxSamples) if _maxSamples.strip() else None

# ===== 이미지 프롬프트 (.env 변경으로 시뮬레이션 가능) =====
IMAGE_INSTRUCTION = os.getenv("IMAGE_INSTRUCTION", "이미지를 보고 필라테스 동작명을 분류하고 한 줄 설명을 제공하세요.")
IMAGE_INPUT       = os.getenv("IMAGE_INPUT",       "동작명과 간단한 설명을 한국어로 답하세요.")

# ===== 시드 고정 =====
SEED = int(os.getenv("SEED", 42))
random.seed(SEED)

print(f"HF 레포         : {HF_DATASET_REPO}")
print(f"이미지 경로     : {IMAGE_ROOT}")
print(f"메타 CSV        : {IMAGE_META_ROOT}")
print(f"텍스트 JSON     : {TEXT_JSON_PATH}")
print(f"클래스당 최대   : {MAX_IMAGE_SAMPLES_PER_CLASS if MAX_IMAGE_SAMPLES_PER_CLASS else '제한 없음'}")

HF 레포         : hyokwan/multimodal_sports_sample
이미지 경로     : ./dataset/04. 멀티모달 샘플/image_samples
메타 CSV        : ./dataset/04. 멀티모달 샘플/image_samples/labels.csv
텍스트 JSON     : ./dataset/04. 멀티모달 샘플/text_samples/sample_data.json
클래스당 최대   : 제한 없음


### 텍스트 데이터 불러오기 및 포맷변환(알파카 포맷)

In [11]:
inDf = pd.read_json(TEXT_JSON_PATH, lines=True)

inDf["modality"] = "text_only"
inDf["image"]    = None
inDf["source"]   = "generated"
inDf["label"]    = ""

alpacaDf = inDf[["modality", "image", "instruction", "input", "output", "source", "label"]]
print(f"텍스트 데이터 로드 완료: {len(alpacaDf)}건")

텍스트 데이터 로드 완료: 30건


In [12]:
alpacaDf

,modality,image,instruction,input,output,source,label
0,text_only,None,다음 운동 동작을 쉽게 설명하시오.,브릿지,바닥에 누워 무릎을 세운 후 엉덩이를 들어 올려 몸을 일직선으로 만드는 동작이다.,generated,
1,text_only,None,다음 운동 동작을 쉽게 설명하시오.,플랭크,팔꿈치와 발끝으로 몸을 지탱하며 몸을 일직선으로 유지하는 코어 운동이다.,generated,
2,text_only,None,다음 운동 동작을 쉽게 설명하시오.,롤업,누운 상태에서 상체를 천천히 들어 올려 앉는 동작이다.,generated,
3,text_only,None,다음 운동 동작을 쉽게 설명하시오.,레그 레이즈,누워서 다리를 들어 올려 복부를 강화하는 운동이다.,generated,
4,text_only,None,다음 운동 동작을 쉽게 설명하시오.,스완,엎드린 상태에서 상체를 들어 올려 척추를 확장하는 운동이다.,generated,
5,text_only,None,다음 상황에 맞는 운동을 추천하시오.,허리 통증이 있는 초보자,브릿지와 가벼운 코어 강화 운동을 추천한다.,generated,
6,text_only,None,다음 상황에 맞는 운동을 추천하시오.,복부 강화 목적,플랭크와 레그 레이즈를 추천한다.,generated,
7,text_only,None,다음 상황에 맞는 운동을 추천하시오.,유연성 향상,스트레칭과 롤다운 동작을 추천한다.,generated,
8,text_only,None,다음 상황에 맞는 운동을 추천하시오.,하체 근력 강화,스쿼트와 런지 동작을 추천한다.,generated,
9,text_only,None,다음 상황에 맞는 운동을 추천하시오.,전신 운동,플랭크와 브릿지를 포함한 복합 운동을 추천한다.,generated,


In [13]:
textDataset = Dataset.from_pandas(alpacaDf)
print(textDataset[0])

{'modality': 'text_only', 'image': None, 'instruction': '다음 운동 동작을 쉽게 설명하시오.', 'input': '브릿지', 'output': '바닥에 누워 무릎을 세운 후 엉덩이를 들어 올려 몸을 일직선으로 만드는 동작이다.', 'source': 'generated', 'label': ''}


### 이미지 데이터 불러오기

In [14]:
df = pd.read_csv(IMAGE_META_ROOT)
print(f"이미지 메타 로드 완료: {len(df)}건  |  클래스: {df['pose_label'].nunique()}종")
df.head()

이미지 메타 로드 완료: 25건  |  클래스: 5종


,image_path,pose_label,level,equipment,gender,source_json,source_type
0,hundred/hundred_001.png,hundred,샘플,mat,female,NaN,synthetic_generated_sample
1,hundred/hundred_002.png,hundred,샘플,mat,female,NaN,synthetic_generated_sample
2,hundred/hundred_003.png,hundred,샘플,mat,female,NaN,synthetic_generated_sample
3,hundred/hundred_004.png,hundred,샘플,mat,female,NaN,synthetic_generated_sample
4,hundred/hundred_005.png,hundred,샘플,mat,female,NaN,synthetic_generated_sample


In [15]:
imageData   = []
classCounts = {}

for i in range(0, len(df)):
    label     = df.loc[i, "pose_label"]
    imagePath = os.path.join(IMAGE_ROOT, df.loc[i, "image_path"])

    # 클래스당 최대 샘플 수 제한 (MAX_IMAGE_SAMPLES_PER_CLASS 설정 시)
    if MAX_IMAGE_SAMPLES_PER_CLASS is not None:
        currentCount = classCounts.get(label, 0)
        if currentCount >= MAX_IMAGE_SAMPLES_PER_CLASS:
            continue
        classCounts[label] = currentCount + 1

    image      = Image.open(imagePath).convert("RGB")
    outputText = f"동작명: {label}\n설명: 이 이미지는 {label} 동작 예시입니다."

    imageData.append({
        "modality":    "image_text",
        "image":       image,
        "instruction": IMAGE_INSTRUCTION,
        "input":       IMAGE_INPUT,
        "output":      outputText,
        "source":      "pilates_dataset",
        "label":       label
    })

print(f"이미지 샘플 수: {len(imageData)}건")

이미지 샘플 수: 25건


In [16]:
imageDataset = Dataset.from_list(imageData)
print(imageDataset)

Dataset({
    features: ['modality', 'image', 'instruction', 'input', 'output', 'source', 'label'],
    num_rows: 25
})


In [17]:
# datasets 버전 업그레이드 후 pandas → large_string / list → string 타입 불일치 해결
# textDataset 피처를 imageDataset 기준으로 캐스팅
textDataset = textDataset.cast(imageDataset.features)

allDataset = concatenate_datasets([imageDataset, textDataset])
print(f"전체 데이터셋: {len(allDataset)}건  (이미지: {len(imageData)}건 + 텍스트: {len(alpacaDf)}건)")

Casting the dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

전체 데이터셋: 55건  (이미지: 25건 + 텍스트: 30건)


In [18]:
allDataset = allDataset.shuffle(seed=SEED)
print("셔플 완료")

셔플 완료


In [19]:
# HuggingFace Hub 업로드 (image 컬럼이 포함되어 파일도 함께 업로드됨)
allDataset.push_to_hub(HF_DATASET_REPO, token=hf_token)
print(f"업로드 완료 → {HF_DATASET_REPO}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

tmptlrw_m8m.parquet:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

업로드 완료 → hyokwan/multimodal_sports_sample
